<a href="https://colab.research.google.com/github/sarfaraziqbal/Hybrid-Telegram-Bot-RAG-Vision-AI-/blob/main/Sarfaraz_AVIVO_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installation

In [ ]:
!pip install sentence-transformers transformers torch torchvision \
python-telegram-bot nest_asyncio pypdf \
langchain-community langchain-text-splitters langchain-openai pillow sqlite-vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.4/745.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core

Upload document

In [ ]:
from google.colab import files

uploaded_files = files.upload()
file_names = list(uploaded_files.keys())

print("Uploaded:", file_names)

Saving 2019BurkovTheHundred-pageMachineLearning_copy.pdf to 2019BurkovTheHundred-pageMachineLearning_copy.pdf
Uploaded: ['2019BurkovTheHundred-pageMachineLearning_copy.pdf']


Load document

In [ ]:
import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader

documents = []

for file in file_names:
    if file.endswith(".txt"):
        loader = TextLoader(file)
    elif file.endswith(".pdf"):
        loader = PyPDFLoader(file)
    else:
        continue

    docs = loader.load()

    #  source metadata
    for d in docs:
        d.metadata["source"] = file

    documents.extend(docs)

Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=550,
    chunk_overlap=50
)

docs = splitter.split_documents(documents)

Embedding model

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SQLite Stoorage

In [ ]:
import sqlite3
import numpy as np

conn = sqlite3.connect("rag.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS docs (
    id INTEGER PRIMARY KEY,
    content TEXT,
    source TEXT,
    embedding BLOB
)
""")

cursor.execute("DELETE FROM docs")

for doc in docs:
    emb = embed_model.encode(doc.page_content).astype(np.float32)

    cursor.execute(
        "INSERT INTO docs (content, source, embedding) VALUES (?, ?, ?)",
        (doc.page_content, doc.metadata["source"], emb.tobytes())
    )

conn.commit()

Retrieval and Reranking With MMR+MQR(Hybrid)

In [ ]:
from sentence_transformers import CrossEncoder
import numpy as np

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# =========================
# Cosine Similarity
# =========================
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# =========================
# 1. BASIC RETRIEVE
# =========================
def retrieve_basic(query, top_k=15):
    query_emb = embed_model.encode(query).astype(np.float32)

    cursor.execute("SELECT content, source, embedding FROM docs")
    rows = cursor.fetchall()

    scored = []
    for content, source, emb_blob in rows:
        emb = np.frombuffer(emb_blob, dtype=np.float32)
        score = cosine_similarity(query_emb, emb)
        scored.append((score, content, source))

    scored.sort(reverse=True)
    return scored[:top_k]


# =========================
# 2. MMR RETRIEVE (DIVERSITY)
# =========================
def retrieve_mmr(query, top_k=15, lambda_param=0.7):
    query_emb = embed_model.encode(query).astype(np.float32)

    cursor.execute("SELECT content, source, embedding FROM docs")
    rows = cursor.fetchall()

    docs = []
    embeddings = []

    for content, source, emb_blob in rows:
        emb = np.frombuffer(emb_blob, dtype=np.float32)
        docs.append((content, source))
        embeddings.append(emb)

    embeddings = np.array(embeddings)

    # similarity with query
    sim_to_query = np.array([
        cosine_similarity(query_emb, emb) for emb in embeddings
    ])

    selected = []
    selected_indices = []

    # pick best first
    first_idx = np.argmax(sim_to_query)
    selected.append((sim_to_query[first_idx], docs[first_idx][0], docs[first_idx][1]))
    selected_indices.append(first_idx)

    # MMR loop
    for _ in range(top_k - 1):
        mmr_scores = []

        for i in range(len(docs)):
            if i in selected_indices:
                continue

            relevance = sim_to_query[i]

            diversity = max([
                cosine_similarity(embeddings[i], embeddings[j])
                for j in selected_indices
            ]) if selected_indices else 0

            mmr_score = lambda_param * relevance - (1 - lambda_param) * diversity
            mmr_scores.append((mmr_score, i))

        if not mmr_scores:
            break

        best_idx = sorted(mmr_scores, reverse=True)[0][1]

        selected.append((sim_to_query[best_idx], docs[best_idx][0], docs[best_idx][1]))
        selected_indices.append(best_idx)

    return selected


# =========================
# 3. MULTI-QUERY RETRIEVE (LLM)
# =========================
def generate_sub_queries(query):
    prompt = f"""
Generate 3 different rephrasings of the query to improve retrieval.

Query: {query}
"""

    response = llm.invoke(prompt).content.strip()

    sub_queries = [q.strip() for q in response.split("\n") if q.strip()]
    return sub_queries[:3]


def retrieve_multi_query(query, top_k=15):
    sub_queries = generate_sub_queries(query)

    all_results = []

    for q in sub_queries:
        results = retrieve_basic(q, top_k=top_k // 3)
        all_results.extend(results)

    # remove duplicates (by content)
    unique = {}
    for score, content, source in all_results:
        if content not in unique:
            unique[content] = (score, content, source)

    return list(unique.values())


# =========================
# 4. FINAL RETRIEVE (HYBRID)
# =========================
def retrieve(query, method="hybrid", top_k=15):

    if method == "basic":
        return retrieve_basic(query, top_k)

    elif method == "mmr":
        return retrieve_mmr(query, top_k)

    elif method == "multi":
        return retrieve_multi_query(query, top_k)

    elif method == "hybrid":
        # Combine MMR + Multi-query
        mmr_results = retrieve_mmr(query, top_k=top_k//2)
        multi_results = retrieve_multi_query(query, top_k=top_k//2)

        combined = mmr_results + multi_results

        # remove duplicates
        unique = {}
        for score, content, source in combined:
            if content not in unique:
                unique[content] = (score, content, source)

        return list(unique.values())[:top_k]

    else:
        return retrieve_basic(query, top_k)


# =========================
# 5. RERANK
# =========================
def rerank(query, retrieved, top_k=5):
    pairs = [[query, item[1]] for item in retrieved]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, retrieved),
        reverse=True
    )

    return [item for _, item in ranked[:top_k]]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

LLM

In [ ]:
from langchain_openai import ChatOpenAI
import os

# environment variables
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3
)

RAG Answer

In [ ]:
# =========================
# CACHE
# =========================
cache = {}

def generate_rag_answer(query, history):

    # =========================
    # CACHE CHECK
    # =========================
    if query in cache:
        return cache[query]["answer"], cache[query]["sources"]

    retrieved = retrieve(query, method="hybrid")
    best_docs = rerank(query, retrieved)

    context = "\n\n".join([doc[1] for doc in best_docs])
    sources = list(set([doc[1] for doc in best_docs]))

    history_text = "\n".join(history)

    prompt = f"""
You are a helpful AI assistant.

Use the provided context to answer the question.
You may expand the answer in your own words to make it complete.

If context is insufficient, you can use general knowledge.

Keep the answer clear and well structured.

History:
{history_text}

Context:
{context}

Question:
{query}
"""

    response = llm.invoke(prompt).content.strip()

    # =========================
    # FALLBACK
    # =========================
    if "I don't know" in response:
        response = llm.invoke(query).content.strip()

    # =========================
    # STORE IN CACHE
    # =========================
    cache[query] = {
        "answer": response,
        "sources": sources
    }

    return response, sources

Image Caption and Tags( BLIP and CLIP)

In [ ]:
from transformers import (
    BlipProcessor, BlipForConditionalGeneration,
    CLIPProcessor, CLIPModel
)
from PIL import Image
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# BLIP (Caption)
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large"
).to(device)

# CLIP (Tags)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)


# Predefined tag vocabulary
CANDIDATE_TAGS = [
    "man", "woman", "person", "portrait",
    "kurta", "shirt", "t-shirt", "jacket", "dress",
    "traditional wear", "ethnic wear", "casual wear",
    "black clothing", "white clothing",

    "indoor", "outdoor", "home", "room",
    "door", "wall", "background",

    "fashion", "posing", "standing", "selfie",
     "Saree","Pant", " Shoe","Chappal"
]


# Caption Generator (BLIP)
def generate_caption(image):
    inputs = blip_processor(image, return_tensors="pt").to(device)

    outputs = blip_model.generate(
        **inputs,
        max_length=40,
        num_beams=5,
        repetition_penalty=1.5,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

    caption = blip_processor.decode(outputs[0], skip_special_tokens=True).strip()

    return caption.capitalize()


# Tag Generator (CLIP)
def generate_tags(image):
    inputs = clip_processor(
        text=CANDIDATE_TAGS,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    outputs = clip_model(**inputs)

    probs = outputs.logits_per_image.softmax(dim=1).detach().cpu().numpy()[0]

    threshold = 0.1

    filtered = [
        (CANDIDATE_TAGS[i], probs[i])
        for i in range(len(probs))
        if probs[i] > threshold
    ]

    # Sort by confidence
    filtered = sorted(filtered, key=lambda x: x[1], reverse=True)

    # Fallback logic
    if len(filtered) == 0:
        top_indices = probs.argsort()[-5:][::-1]
        tags = [CANDIDATE_TAGS[i] for i in top_indices]
    else:
        tags = [tag for tag, _ in filtered[:5]]

    return tags


# FINAL FUNCTION
def generate_caption_and_tags(image_path):
    image = Image.open(image_path).convert("RGB")

    caption = generate_caption(image)
    tags = generate_tags(image)

    return caption, tags

preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Memory

In [ ]:
user_memory = {}

def update_memory(user_id, entry):
    if user_id not in user_memory:
        user_memory[user_id] = []

    user_memory[user_id].append(entry)

    # keep last 3
    user_memory[user_id] = user_memory[user_id][-3:]

Telegram Bot

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes
import os

TOKEN = "8775248785:AAGholW6ijxGkzYfRA12Rqsqv7W0MIR8tSU"

# =========================
# STATE + MEMORY
# =========================
user_state = {}
user_memory = {}

def update_memory(user_id, entry):
    if user_id not in user_memory:
        user_memory[user_id] = []

    user_memory[user_id].append(entry)

    # keep last 3 (as per assignment)
    user_memory[user_id] = user_memory[user_id][-3:]


def get_last_interaction(user_id):
    if user_id not in user_memory or len(user_memory[user_id]) == 0:
        return None
    return user_memory[user_id][-1]


# =========================
# /start
# =========================
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "🤖 Hybrid RAG + Vision Bot\n\n"
        "/ask <question>\n"
        "/image → upload image\n"
        "/help"
    )


# =========================
# /help
# =========================
async def help_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
    "Commands:\n"
    "/ask What is AI?\n"
    "/image → upload image\n"
    "/summarize → summarize last response"
    "/help Ask for help from bot in keywords"
)


# =========================
# /ask → STREAMING + RAG + SOURCES + HISTORY
# =========================
async def ask(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    query = " ".join(context.args)

    if not query:
        await update.message.reply_text("⚠️ Provide question")
        return

    msg = await update.message.reply_text("🤔 Thinking...")

    # =========================
    # SPECIAL CASE: PREVIOUS QUESTION
    # =========================
    if "previous question" in query.lower() or "last question" in query.lower():
        last = get_last_interaction(user_id)

        if not last:
            await msg.edit_text("I don't have any previous interaction.")
            return

        if last["type"] == "text":
            await msg.edit_text(f"🧠 Your previous question was:\n👉 {last['query']}")

        elif last["type"] == "image":
            await msg.edit_text(
                f"🖼 You shared an image.\n"
                f"📌 It looked like: {last['caption']}"
            )
        return

    # =========================
    # PREPARE HISTORY FOR RAG
    # =========================
    history = [
        f"Q: {h['query']}\nA: {h['response']}"
        for h in user_memory.get(user_id, [])
        if h["type"] == "text"
    ]

    # =========================
    # RAG CALL
    # =========================
    answer, sources = generate_rag_answer(query, history)

    # =========================
    # 🔥 STREAMING SIMULATION
    # =========================
    display_text = ""
    last_update_len = 0

    for char in answer:
        display_text += char

        # update every ~120 chars
        if len(display_text) - last_update_len > 120:
            try:
                await msg.edit_text(display_text[:4000])
                last_update_len = len(display_text)
            except:
                pass

    # =========================
    # FINAL OUTPUT (WITH SOURCES)
    # =========================
    sources_text = "\n".join(set(sources))

    final_text = (
        f"💡 Answer:\n{answer}\n\n"
        f"📚 Sources:\n{sources_text}"
    )

    await msg.edit_text(final_text[:4000])

    # =========================
    # SAVE MEMORY
    # =========================
    update_memory(user_id, {
        "type": "text",
        "query": query,
        "response": answer
    })


# =========================
# /image → set mode
# =========================
async def image_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_state[update.effective_user.id] = "image"
    await update.message.reply_text("📷 Upload an image now")


# =========================
# HANDLE IMAGE → + MEMORY
# =========================
async def handle_image(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id

    if user_state.get(user_id) != "image":
        return

    photo = update.message.photo[-1]
    file = await photo.get_file()

    file_path = "input.jpg"
    await file.download_to_drive(file_path)

    msg = await update.message.reply_text("⏳ Processing image...")

    try:
        await msg.edit_text("🧠 Generating caption...")
        caption, tags = generate_caption_and_tags(file_path)

        final_text = (
            f"🖼 Caption:\n{caption}\n\n"
            f"🏷 Tags:\n{', '.join(tags)}"
        )

        await msg.edit_text(final_text[:4000])

        # SAVE MEMORY
        update_memory(user_id, {
            "type": "image",
            "caption": caption,
            "tags": tags
        })

    except Exception as e:
        await msg.edit_text(f"❌ Error: {str(e)}")

    user_state[user_id] = None



# =========================
# SUMMARIZE FUNCTION
# =========================
async def summarize(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id

    last = get_last_interaction(user_id)

    if not last:
        await update.message.reply_text("No previous interaction to summarize.")
        return

    msg = await update.message.reply_text("🧠 Summarizing...")

    # =========================
    # TEXT SUMMARY
    # =========================
    if last["type"] == "text":
        text = last["response"]

        prompt = f"""
Summarize the following text in a short and clear way:

{text}
"""

        summary = llm.invoke(prompt).content.strip()

        await msg.edit_text(f"📄 Summary:\n{summary}")

    # =========================
    # IMAGE SUMMARY
    # =========================
    elif last["type"] == "image":
        caption = last["caption"]
        tags = ", ".join(last["tags"])

        summary = f"Image showing {caption}. Key elements: {tags}."

        await msg.edit_text(f"🖼 Summary:\n{summary}")

# =========================
# RUN BOT
# =========================
app = ApplicationBuilder().token(TOKEN).build()

app.add_handler(CommandHandler("start", start))
app.add_handler(CommandHandler("help", help_cmd))
app.add_handler(CommandHandler("ask", ask))
app.add_handler(CommandHandler("image", image_cmd))
app.add_handler(CommandHandler("summarize", summarize))
app.add_handler(MessageHandler(filters.PHOTO, handle_image))

print("✅ Hybrid Bot Running")
app.run_polling()

✅ Hybrid Bot Running


RuntimeError: Cannot close a running event loop